# Lean-RGC v0.2 Quickstart

This notebook runs the Lean-RGC scaffold in dry-run mode, then shows the commands to switch to a real Lean/Lake project.

In [ ]:
import os, zipfile, pathlib, json, textwrap, subprocess, sys
ROOT = pathlib.Path('/content/lean_rgc_v2')
ROOT.mkdir(parents=True, exist_ok=True)
print(ROOT)

## Upload / unzip package

Upload `lean_rgc_automation_stack_v2.zip` or adjust `ZIP_PATH`.

In [ ]:
ZIP_PATH = '/content/lean_rgc_automation_stack_v2.zip'
if pathlib.Path(ZIP_PATH).exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)
else:
    print('Upload the zip to', ZIP_PATH)
PKG = next(ROOT.glob('lean_rgc_stack'), ROOT)
print('package root:', PKG)

In [ ]:
%pip install -q -e {PKG}

## Dry-run audit and response learner

In [ ]:
RUN = ROOT / 'runs' / 'dry'
RUN.mkdir(parents=True, exist_ok=True)
!lean-rgc audit --tasks {PKG}/examples/minimal_theorems.jsonl --actions {PKG}/examples/tactic_templates.jsonl --out {RUN}/audit --dry-run --max-actions 5
!lean-rgc train-response --responses {RUN}/audit/responses.jsonl --actions {PKG}/examples/tactic_templates.jsonl --out {RUN}/resp_model.json
!lean-rgc quotient --responses {RUN}/audit/responses.jsonl --out {RUN}/components.jsonl
!lean-rgc carrier-generate --defects {RUN}/audit/defects.jsonl --out {RUN}/carrier_generated.jsonl
!lean-rgc carrier-coker --defects {RUN}/audit/defects.jsonl --actions {PKG}/examples/tactic_templates.jsonl --out {RUN}/carrier_coker.jsonl

## RGC guided dry-run search

In [ ]:
!lean-rgc run-search --tasks {PKG}/examples/minimal_theorems.jsonl --out {RUN}/search --response-model {RUN}/resp_model.json --dry-run --max-steps 3 --max-candidates 5 --carrier-budget 10
print((RUN/'search'/'trajectory_summary.json').read_text())

## Create a Lean/Lake template

In [ ]:
!lean-rgc init-lake --out {RUN}/lake_template --no-mathlib
!find {RUN}/lake_template -maxdepth 3 -type f -print

## Real Lean mode

After creating or entering a real Lake project, replace `--dry-run` with `--workdir /path/to/lake_project --lean-cmd "lake env lean"`.